# Relational Joins & Dataset Stacking (`join` & `concat`)

`pytae` provides high-performance relational joins and dataset concatenation powered natively by Polars' SIMD vector core.
- **Symmetrical Strategy Support**: `how='inner'`, `'left'`, `'right'`, `'full'`, `'semi'`, `'anti'`, `'cross'`
- **Colon Key Shorthand**: `on='client_id:id'` directly maps differing column keys without typing separate `left_on` / `right_on`
- **Cardinality Validation**: `validate='1:1'`, `'1:m'`, `'m:1'` verifies key uniqueness to prevent silent row explosion
- **Row Stacking**: `pt.concat([df1, df2])` and `df1.concat(df2)` for appending matching datasets
- **Lazy Execution Pushdown**: Preserves query pushdowns when joining lazy scans (`pt.scan_parquet`)

In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt
import polars as pl

In [2]:
# Sample datasets for joining
orders = pt.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "customer_id": [1, 2, 1, 3, 99],  # customer 99 does not exist in customers
    "amount": [250.0, 120.0, 310.0, 85.0, 400.0],
    "item": ["Laptop", "Mouse", "Monitor", "Keyboard", "Desk"]
})

customers = pt.DataFrame({
    "customer_id": [1, 2, 3, 4],  # customer 4 has no orders
    "name": ["Alice", "Bob", "Charlie", "Diana"],
    "region": ["West", "East", "West", "South"]
})

print("Orders:")
display(orders)
print("Customers:")
display(customers)

Orders:


order_id,customer_id,amount,item
i64,i64,f64,str
101,1,250.0,"""Laptop"""
102,2,120.0,"""Mouse"""
103,1,310.0,"""Monitor"""
104,3,85.0,"""Keyboard"""
105,99,400.0,"""Desk"""


Customers:


customer_id,name,region
i64,str,str
1,"""Alice""","""West"""
2,"""Bob""","""East"""
3,"""Charlie""","""West"""
4,"""Diana""","""South"""


## 1. Inner Join (`how='inner'`)
Keeps only rows with matching keys in both datasets:

In [3]:
orders.join(customers, on="customer_id", how="inner")

order_id,customer_id,amount,item,name,region
i64,i64,f64,str,str,str
101,1,250.0,"""Laptop""","""Alice""","""West"""
102,2,120.0,"""Mouse""","""Bob""","""East"""
103,1,310.0,"""Monitor""","""Alice""","""West"""
104,3,85.0,"""Keyboard""","""Charlie""","""West"""


## 2. Left Outer Join (`how='left'`)
Preserves all orders; unmatched customers (e.g. customer 99) receive `null`:

In [4]:
orders.join(customers, on="customer_id", how="left")

order_id,customer_id,amount,item,name,region
i64,i64,f64,str,str,str
101,1,250.0,"""Laptop""","""Alice""","""West"""
102,2,120.0,"""Mouse""","""Bob""","""East"""
103,1,310.0,"""Monitor""","""Alice""","""West"""
104,3,85.0,"""Keyboard""","""Charlie""","""West"""
105,99,400.0,"""Desk""",null,null


## 3. Semi Join (`how='semi'`) & Anti Join (`how='anti'`)
- **Semi Join**: Filters rows in the left dataset that have a matching key in the right table (without appending right columns).
- **Anti Join**: Identifies orphan records in the left dataset that have **no** match in the right table.

In [5]:
# Semi join: orders placed by known customers
orders.join(customers, on="customer_id", how="semi")

order_id,customer_id,amount,item
i64,i64,f64,str
101,1,250.0,"""Laptop"""
102,2,120.0,"""Mouse"""
103,1,310.0,"""Monitor"""
104,3,85.0,"""Keyboard"""


In [6]:
# Anti join: customers who have NEVER placed an order
customers.join(orders, on="customer_id", how="anti")

customer_id,name,region
i64,str,str
4,"""Diana""","""South"""


## 4. Differing Key Names (Colon Shorthand `left_key:right_key`)

When keys have different names, use `on="left_key:right_key"` instead of separate `left_on` and `right_on`:

In [7]:
users = pt.DataFrame({
    "uid": [1, 2, 3],
    "email": ["alice@work.com", "bob@work.com", "charlie@work.com"]
})

# Join orders.customer_id with users.uid
orders.join(users, on="customer_id:uid", how="inner")

order_id,customer_id,amount,item,uid,email
i64,i64,f64,str,i64,str
101,1,250.0,"""Laptop""",1,"""alice@work.com"""
102,2,120.0,"""Mouse""",2,"""bob@work.com"""
103,1,310.0,"""Monitor""",1,"""alice@work.com"""
104,3,85.0,"""Keyboard""",3,"""charlie@work.com"""


## 5. Join Cardinality Validation (`validate=`)

Guard against silent duplication by validating key relationships:
- `validate="1:1"`: Keys must be unique in both left and right frames.
- `validate="m:1"`: Keys must be unique in the right frame.
- `validate="1:m"`: Keys must be unique in the left frame.

In [8]:
# Validate m:1 (many orders per unique customer)
orders.join(customers, on="customer_id", how="inner", validate="m:1")

order_id,customer_id,amount,item,name,region
i64,i64,f64,str,str,str
101,1,250.0,"""Laptop""","""Alice""","""West"""
102,2,120.0,"""Mouse""","""Bob""","""East"""
103,1,310.0,"""Monitor""","""Alice""","""West"""
104,3,85.0,"""Keyboard""","""Charlie""","""West"""


## 6. Dataset Stacking (`pt.concat` & `df.concat`)

Stack multiple datasets sharing common schemas into a single unified table:

In [9]:
sales_q1 = pt.DataFrame({"quarter": ["Q1", "Q1"], "revenue": [1000, 1500]})
sales_q2 = pt.DataFrame({"quarter": ["Q2", "Q2"], "revenue": [1200, 1800]})

# Functional concat
total_sales = pt.concat([sales_q1, sales_q2])
display(total_sales)

# Method chaining equivalent
display(sales_q1.concat(sales_q2))

quarter,revenue
str,i64
"""Q1""",1000
"""Q1""",1500
"""Q2""",1200
"""Q2""",1800


quarter,revenue
str,i64
"""Q1""",1000
"""Q1""",1500
"""Q2""",1200
"""Q2""",1800
